# Notebook 01: Análise Geral do Acervo Booklog

O objetivo é compreender a distribuição básica do catálogo de livros disponíveis no dataset.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import numpy as np
import scipy.stats as st

sns.set_theme(style="whitegrid")

df = pd.read_parquet("datasets/books.parquet")

df.head(10)

,author,bookformat,desc,genre,img,isbn,isbn13,link,pages,rating,reviews,title,totalratings
0,Laurence M. Hauptman,Hardcover,Reveals that several hundred thousand Indians ...,"History,Military History,Civil War,American Hi...",https://i.gr-assets.com/images/S/compressed.ph...,002914180X,9.78E+12,https://goodreads.com/book/show/1001053.Betwee...,0,3.52,5,Between Two Fires: American Indians in the Civ...,33
1,"Charlotte Fiell,Emmanuelle Dirix",Paperback,Fashion Sourcebook - 1920s is the first book i...,"Couture,Fashion,Historical,Art,Nonfiction",https://i.gr-assets.com/images/S/compressed.ph...,1906863482,9.78E+12,https://goodreads.com/book/show/10010552-fashi...,576,4.51,6,Fashion Sourcebook 1920s,41
2,Andy Anderson,Paperback,The seminal history and analysis of the Hungar...,"Politics,History",https://i.gr-assets.com/images/S/compressed.ph...,948984147,9.78E+12,https://goodreads.com/book/show/1001077.Hungar...,124,4.15,2,Hungary 56,26
3,Carlotta R. Anderson,Hardcover,"""All-American Anarchist"" chronicles the life a...","Labor,History",https://i.gr-assets.com/images/S/compressed.ph...,814327079,9.78E+12,https://goodreads.com/book/show/1001079.All_Am...,324,3.83,1,All-American Anarchist: Joseph A. Labadie and ...,6
5,Jeffrey Pfeffer,Hardcover,Why is common sense so uncommon when it comes ...,"Business,Leadership,Romance,Historical Romance...",https://i.gr-assets.com/images/S/compressed.ph...,875848419,9.78E+12,https://goodreads.com/book/show/1001090.The_Hu...,368,3.73,7,The Human Equation: Building Profits by Puttin...,119
6,Jeffrey Pfeffer,Paperback,"""Competitive Advantage Through People"" explore...","Business,Leadership,Business,Management,Romanc...",https://i.gr-assets.com/images/S/compressed.ph...,087584717X,9.78E+12,https://goodreads.com/book/show/1001092.Compet...,304,3.65,1,Competitive Advantage Through People: Unleashi...,20
7,Edward Joesting,Paperback,"""Even if you know Hawaiian history you will fi...","History,Nonfiction",https://i.gr-assets.com/images/S/compressed.ph...,393009076,9.78E+12,https://goodreads.com/book/show/1001126.Hawaii,353,3.93,2,Hawaii: An Uncommon History,15
9,"B. Alan Wallace,Dalai Lama XIV",Hardcover,"Discover your personal path to bliss,""This boo...","Religion,Buddhism,Philosophy,Spirituality,Psyc...",https://i.gr-assets.com/images/S/compressed.ph...,047146984X,9.78E+12,https://goodreads.com/book/show/100114.Genuine...,256,4.10,7,Genuine Happiness: Meditation as the Path to F...,133
11,Brian Morris,Paperback,"In this important, scholarly and wide-ranging ...","Anthropology,Religion,Nonfiction,Academic,Soci...",https://i.gr-assets.com/images/S/compressed.ph...,052133991X,9.78E+12,https://goodreads.com/book/show/1001204.Anthro...,382,3.69,3,Anthropological Studies of Religion: An Introd...,67
12,Graham Purchase,Paperback,This book outlines the history of our slow ali...,"Biology,Ecology,Nonfiction,Politics",https://i.gr-assets.com/images/S/compressed.ph...,1551640260,9.78E+12,https://goodreads.com/book/show/1001209.Anarch...,216,3.59,3,Anarchism And Ecology,29


# 1. Os gêneros mais presentes no banco de dados.

In [2]:
top_categorias = df['genre'].value_counts().head(10).reset_index()
top_categorias.columns = ['Categoria', 'Quantidade']

fig = px.bar(
    top_categorias, 
    x='Quantidade', 
    y='Categoria',
    orientation='h',
    title='<b>Top 10 Gêneros mais populares no dataset</b>',
    color='Quantidade',
    color_continuous_scale='Spectral',
    labels={'Quantidade': 'Número de Livros no Acervo', 'Categoria': 'Gênero'}
)

fig.update_layout(
    title_font_size=16,
    xaxis_title='Número de Livros no Acervo',
    yaxis_title='Categoria',
    yaxis={'categoryorder': 'total ascending'},
    coloraxis_showscale=False,
    template='plotly_white',
    width=800,
    height=500
)

fig.show()

# 2. Autores com mais livros no acervo.
O objetivo é entender se o catálogo é pulverizado ou concentrado em grandes nomes da literatura.

In [3]:
top_autores = df['author'].value_counts().head(10).reset_index()
top_autores.columns = ['Autor', 'Quantidade']

fig = px.bar(
    top_autores, 
    x='Quantidade', 
    y='Autor',
    orientation='h',
    title='<b>Top 10 Autores com Mais Livros no Catálogo</b>',
    color='Quantidade',
    color_continuous_scale='Spectral',
    labels={'Quantidade': 'Quantidade de Livros', 'Autor': 'Autor'}
)

fig.update_layout(
    title_font_size=16,
    xaxis_title='Quantidade de Livros',
    yaxis_title='Autor',
    yaxis={'categoryorder': 'total ascending'},
    coloraxis_showscale=False,
    template='plotly_white',
    width=800,
    height=500
)

fig.show()

# 3. Distribuição das Notas
O objetivo é analisar a curva de distribuição que nos mostrará se a maior parte dos livros possui notas medianas, ou se existe uma tendência a notas altas ou baixas.

In [4]:
df_valid = df[df['rating'].notna()]
x_grid = np.linspace(df_valid['rating'].min(), df_valid['rating'].max(), 300)
y_kde = st.gaussian_kde(df_valid['rating'])(x_grid)

counts, bins = np.histogram(df_valid['rating'], bins=30)
bin_width = bins[1] - bins[0]
y_scale = len(df_valid) * bin_width
y_kde_scaled = y_kde * y_scale

fig = px.histogram(
    df_valid, 
    x='rating', 
    nbins=30, 
    title='<b>Distribuição de Notas do Acervo</b>',
    color_discrete_sequence=["#537EDB"], 
    labels={'rating': 'Nota Média (0 a 5)', 'count': 'Quantidade de Livros'}
)

fig.add_scatter(
    x=x_grid, 
    y=y_kde_scaled, 
    mode='lines', 
    line=dict(color='#FF10F0', width=3),
    showlegend=False,
    name='Curva KDE'
)

fig.update_traces(
    marker_line_width=0,
    selector=dict(type='histogram')
)

fig.update_layout(
    title_font_size=16,
    xaxis_title='Nota Média (0 a 5)',
    yaxis_title='Quantidade de Livros',
    template='plotly_white',
    width=800,
    height=500,
    hovermode='x unified'
)

fig.show()

# 4. Distribuição do Número de Páginas

Analise dos tamanhos das obras presentes no dataset, se os livros tendem a ser curtos, ou obras grandes.

In [5]:
df_valid = df[(df['pages'].notna()) & (df['pages'] > 0) & (df['pages'] <= 1000)]

counts, bins = np.histogram(df_valid['pages'], bins=50, range=(0, 1000))
bin_width = bins[1] - bins[0]
y_scale = len(df_valid) * bin_width

df_valid = df[(df['pages'].notna()) & (df['pages'] > 0) & (df['pages'] <= 1000)]

counts, bins = np.histogram(df_valid['pages'], bins=50, range=(0, 1000))
bin_width = bins[1] - bins[0]
y_scale = len(df_valid) * bin_width

x_grid = np.linspace(0, 1000, 300)
y_kde = st.gaussian_kde(df_valid['pages'])(x_grid)
y_kde_scaled = y_kde * y_scale


fig = px.histogram(
    df_valid, 
    x='pages', 
    nbins=50, 
    range_x=[0, 1000],
    title='<b>Perfil de Extensão dos Livros</b>',
    color_discrete_sequence=["#E0953F"], 
    labels={'pages': 'Número de Páginas', 'count': 'Quantidade de Livros'}
)

fig.add_scatter(
    x=x_grid, 
    y=y_kde_scaled, 
    mode='lines', 
    line=dict(color='#6A0DAD', width=3), 
    showlegend=False,
    name='Curva KDE'
)

fig.update_traces(
    marker_line_width=0,
    selector=dict(type='histogram')
)

fig.update_layout(
    title_font_size=14,
    xaxis_title='Número de Páginas',
    yaxis_title='Quantidade de Livros',
    template='plotly_white',
    width=800,
    height=500,
    hovermode='x unified'
)

fig.show()